# Gorilla: a Kubernetes API-call pilot
Follow **dataset curation → retrieval → API-call generation → AST evaluation** from [paper.pdf](../paper.pdf), §§3.1–3.3, pp. 4–6.

This is an adaptation using Gemma and Qwen. **I did not reproduce Gorilla’s fine-tuning or Retriever-Aware Training (RAT).**

## 1. Dataset curation
30 API documentation records come from Kubernetes v1.35.2 OpenAPI, checked against live discovery. The 20 user-provided instructions were revised and verified using reads/server dry-runs. Reference answers are evaluator-only. Run from the project root or `notebooks/`; dependencies: `scikit-learn`, `ipykernel`.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import Markdown, display

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/apis.json").exists())
sys.path.insert(0, str(root / "notebooks"))

def load_json(path):
    return json.loads((root / path).read_text())

def show_table(headers, rows):
    lines = [headers, ["---"] * len(headers), *rows]
    display(Markdown("\n".join("| " + " | ".join(map(str, row)) + " |" for row in lines)))

def rate(count, total):
    return f"{count}/{total} ({count / total:.0%})" if total else "N/A"

apis = load_json("data/apis.json")["apis"]
queries = load_json("data/queries.json")["requests"]
api_ids = np.array([api["api_name"] for api in apis])
assert len(apis) == 30 and len(queries) == 20
assert all(set(query) == {"request_id", "input"} for query in queries)
print(f"{len(apis)} APIs; {len(queries)} instructions")

30 APIs; 20 instructions


## 2. Retrieval
The paper uses BM25 or GPT-Index with top-1 documents (pp. 6–7). My baseline uses **TF-IDF + cosine similarity**, retrieving the top three API descriptions. Fit only on documentation; use reference labels after ranking.

In [2]:
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
api_vectors = vectorizer.fit_transform([api["description"] for api in apis])
query_vectors = vectorizer.transform([query["input"] for query in queries])
scores = cosine_similarity(query_vectors, api_vectors)
ranked = np.argsort(-scores, axis=1, kind="stable")
top3 = api_ids[ranked[:, :3]].tolist()

answers = load_json("data/evaluation/answers.json")["answers"]
gold = {answer["request_id"]: answer["correct_api"] for answer in answers}
assert set(gold) == {query["request_id"] for query in queries}
for k in [1, 3]:
    hits = sum(gold[query["request_id"]] in candidates[:k]
               for query, candidates in zip(queries, top3))
    print(f"Top-{k} retrieval accuracy: {rate(hits, len(queries))}")

Top-1 retrieval accuracy: 10/20 (50%)
Top-3 retrieval accuracy: 17/20 (85%)


## 3. API-call generation
Following the inference pattern on p. 5, send the instruction with retrieved API documentation and SDK argument metadata. Request **one Python API call** or `NO_MATCHING_API`; never send reference answers.

Both models use temperature 0. Gemma allows 4,096 output tokens; Qwen allows 3,310, with thinking disabled. Saved responses are reused. `RUN_NEW_REQUESTS=True` permits only missing requests; failures are retained without retries. Connection and credential handling live in helpers.

In [3]:
from gorilla_google import google_request, read_google_key
from gorilla_eval import run_generation, evaluate_run
from gorilla_vllm import run_qwen, verify_same_inputs

RUN_NEW_REQUESTS = False
output_dir = root / "data/evaluation"
connection_file = root / "notebooks/vllm_connection.local.json"
secret_values = [read_google_key()] if RUN_NEW_REQUESTS else []

gemma = run_generation(queries, top3, apis, "gemma-4-31b-it", google_request,
                       output_dir / "gemma_4_31b_generation.json", allow_requests=RUN_NEW_REQUESTS)
qwen = run_qwen(queries, top3, apis, connection_file,
                output_dir / "qwen3_8b_generation.json", allow_requests=RUN_NEW_REQUESTS)
verify_same_inputs(gemma, qwen)
runs = [gemma, qwen]
print("Same instructions, prompt template, and top-three APIs: verified.")

Same instructions, prompt template, and top-three APIs: verified.


## 4. AST evaluation
The paper uses **AST sub-tree matching** for API identity and selected arguments (pp. 5–6). My adaptation checks one allowed SDK call and its verified arguments; it does **not execute code**. Correct-call rate is reference-call agreement, not runtime success.

Report all-request and API-success-only rates. Provider/transport errors are separate from wrong calls.

In [4]:
evaluations = [evaluate_run(run, answers, apis) for run in runs]
rows = []
for evaluation in evaluations:
    result = evaluation["summary"]
    rows.append([evaluation["model"], rate(result["correct_calls"], result["queries"]),
                 rate(result["correct_calls"], result["api_successes"]), result["api_errors"]])
show_table(["Model", "Correct calls / all requests", "Among API successes", "API errors"], rows)

| Model | Correct calls / all requests | Among API successes | API errors |
| --- | --- | --- | --- |
| gemma-4-31b-it | 3/20 (15%) | 3/4 (75%) | 16 |
| Qwen3-8B | 4/20 (20%) | 4/20 (20%) | 0 |

## 5. API-removal experiment — my extension
For each of 10 fixed instructions, independently remove its correct API from the original 30 and refit retrieval on 29. Reuse each model and the prompt template. Qwen’s output cap is 3,287 here: a 23-token reduction fixed before generation to fit context.

A refusal is a complete response equal to `NO_MATCHING_API` after trimming whitespace. API errors never count as refusals. This is **not** the paper’s training/retrieval ablation (Table 2, p. 8).

In [5]:
from gorilla_refusal import build_trials, run_refusal, evaluate_refusal

refusal_dir = output_dir / "refusal"
plan = load_json("data/evaluation/refusal/plan.json")
registry = load_json("data/evaluation/refusal/kubernetes_api_registry.json")
trials = build_trials(apis, queries, answers, plan["query_ids"])
assert trials == load_json("data/evaluation/refusal/trials.json")["trials"]
print("Fixed instructions:", ", ".join(plan["query_ids"]))

removal_evaluations = []
for short, model in [("gemma", "gemma-4-31b-it"), ("qwen", "Qwen3-8B")]:
    run = run_refusal(trials, model, plan["model_settings"][model],
                      refusal_dir / f"{short}_generation.json", google_request=google_request,
                      connection_file=connection_file, allow_requests=RUN_NEW_REQUESTS,
                      forbidden_values=secret_values)
    removal_evaluations.append(evaluate_refusal(run, trials, answers, apis, registry))

Fixed instructions: req_002, req_004, req_007, req_009, req_010, req_012, req_013, req_014, req_015, req_019


In [6]:
rows = []
for evaluation in removal_evaluations:
    result = evaluation["summary"]
    counts = result["outcome_counts"]
    rows.append([evaluation["model"], rate(result["exact_refusals"], result["queries"]),
                 rate(result["exact_refusals"], result["api_successes"]), result["api_errors"],
                 counts.get("UNKNOWN_API", 0), counts.get("REAL_BUT_UNAVAILABLE", 0)])
show_table(["Model", "Refusals / all requests", "Among API successes", "API errors",
            "Unknown API", "Real but unavailable"], rows)

| Model | Refusals / all requests | Among API successes | API errors | Unknown API | Real but unavailable |
| --- | --- | --- | --- | --- | --- |
| gemma-4-31b-it | 1/10 (10%) | 1/1 (100%) | 9 | 0 | 0 |
| Qwen3-8B | 3/10 (30%) | 3/10 (30%) | 0 | 1 | 2 |

## 6. Hallucination and actual failures
The paper calls an API unmatched anywhere in its database a **hallucination** (p. 6). My taxonomy checks the pinned Kubernetes SDK/REST registry: **unknown API** is an unsupported name; **real but unavailable** is a real API absent from the three provided cards. These are distinct from wrong arguments or invalid syntax.

Two exact observed unsupported SDK calls:

In [7]:
failures = load_json("data/evaluation/refusal/failure_examples.json")["examples"]
examples = {("baseline", "req_005"), ("withheld", "req_019")}
for failure in failures:
    if (failure["condition"], failure["request_id"]) in examples and failure["taxonomy"]["outcome"] == "UNKNOWN_API":
        print(failure["model"], failure["condition"], failure["request_id"])
        print(failure["response_text"], "\n")

Qwen3-8B baseline req_005
list_resource_v1_device_class(resource_version="0") 

Qwen3-8B withheld req_019
scheduling_v1.read_namespaced_priority_class(name="system-cluster-critical", namespace="") 



## 7. Saved evidence and limits
Save metrics, model settings, prompts, recorded responses, and actual failure examples. Baseline prompts were reconstructed and hash-checked; missing raw responses were not invented.

This small, selected pilot does not establish generalization. Gemma’s many API errors prevent a reliable model ranking. An earlier Qwen transport failure run is preserved separately and excluded ([details](../data/evaluation/qwen3_8b_transport_fix.json)).

Original raw cluster-check files are unavailable; historical reference summaries remain.

In [8]:
from gorilla_report import save_experiment

experiment = save_experiment(root, forbidden_values=secret_values)
for evaluation in evaluations:
    assert evaluation["summary"]["correct_calls"] == experiment["baseline"][evaluation["model"]]["correct_calls"]
for evaluation in removal_evaluations:
    assert evaluation["summary"] == experiment["withheld"][evaluation["model"]]
print("Saved evidence: data/evaluation/refusal/")

Saved evidence: data/evaluation/refusal/


[Summary](../data/evaluation/refusal/summary.json) · [Prompts](../data/evaluation/refusal/trials.json) · [Failures](../data/evaluation/refusal/failure_examples.json) · [Full protocol](../data/evaluation/refusal/README.md)